In [ ]:
%pip install --quiet --upgrade langchain-text-splitters langchain-community faiss-cpu
%pip install -qU "langchain[openai]"
!pip install langchain-google-genai
%pip install --upgrade google-ai-generativelanguage>=0.6.18,<0.7.0 langchain-google-genai
!pip install -q transformers accelerate

In [ ]:
import os
import pandas as pd
import csv
import json
import torch
import time
import torch.nn.functional as F
import google.generativeai as genai
from datetime import datetime
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document
from typing import List, TypedDict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from huggingface_hub import login

os.environ["USER_AGENT"] = "Mozilla/5.0 (compatible; MyLangChainBot/1.0; +http://mywebsite.com/bot)"
os.environ["LANGCHAIN_TRACING_V2"]="false"
os.environ["LANGCHAIN_API_KEY"]="YOUR_LANGCHAIN_API_KEY"
os.environ["OPENAI_API_KEY"]="YOUR_OPENAI_API_KEY"
#os.environ["OPENROUTER_API_KEY"]="YOUR_OPENROUTER_API_KEY"
os.environ["HF_TOKEN"]="YOUR_HF_TOKEN"
os.environ["MISTRAL_AI_API_KEY"]="YOUR_MISTRAL_API_KEY"
os.environ["GEMINI_API_KEY"]="YOUR_GEMINI_API_KEY"
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

CSV_PROGRESS_OUTPUT = "/content/drive/MyDrive/Colab Notebooks/mod_progress_output.csv"
CSV_MESSAGES_OUTPUT = "/content/drive/MyDrive/Colab Notebooks/mod_progress_messages_output.csv"
FAISS_INDEX_1 = "/content/drive/MyDrive/Colab Notebooks/faiss_index_1"
CSV_PATH_1 = "/content/drive/MyDrive/Colab Notebooks/Stats/results_summary_1.csv"
JSON_OUTPUT = "/content/drive/MyDrive/Colab Notebooks/output.json"

In [ ]:
# Functions

# Prompt function
def build_prompt_template(response_type="default"):
    if response_type == "boolean":
        template = (
            "Rispondi alla seguente domanda in italiano con 'True' o 'False', "
            "usando solo il contesto fornito. Se l'informazione non è presente, rispondi 'Non lo so'.\n\n"
            "Contesto:\n{context}\n\nDomanda:\n{question}\n\nRisposta:"
        )
    else:
        template = (
            "Rispondi alla seguente domanda in italiano, utilizzando solo le informazioni fornite nel contesto. "
            "Se la risposta non è presente nel contesto, dì semplicemente 'Non lo so'.\n\n"
            "Contesto:\n{context}\n\nDomanda:\n{question}\n\nRisposta:"
        )
    return PromptTemplate.from_template(template)

# Reranking function
def rerank(query, docs, top_n=3):
    pairs = [(query, doc.page_content) for doc in docs]
    texts = [f"{q} </s> {d}" for q, d in pairs]
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits.squeeze()

    if len(pairs) == 1:
        logits = logits.unsqueeze(0)

    scores = F.softmax(logits, dim=0)
    reranked_docs = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in reranked_docs[:top_n]]

#Define print function
def printQA(question, answer):
    print(f"Domanda:\n{question}\n")
    print("\n" + "=" * 50 + "\n")
    print("Contesto trovato:\n")
    print(docs_content)
    print("\n" + "=" * 50 + "\n")
    print("Risposta:\n")
    print(answer.content)
    print("\n" + "=" * 50 + "\n")

# Comparison function
def compare_answers(generated, correct):
    return generated.strip().lower() == correct.strip().lower()

# % of success
def calculate_success_rate(generated_answers, correct_answers):
    assert len(generated_answers) == len(correct_answers), "Numero di risposte deve coincidere"
    matches = sum(compare_answers(gen, cor) for gen, cor in zip(generated_answers, correct_answers))
    return matches / len(correct_answers) * 100

# Reading file function
def read_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f.readlines() if line.strip()]

# Semantic similarity function
def semantic_similarity(answer1, answer2):
    vec1 = embedding_model.embed_query(answer1)
    vec2 = embedding_model.embed_query(answer2)
    return cosine_similarity([vec1], [vec2])[0][0]

# Variables
llm_models = {
    # "gpt-4.1": ChatOpenAI(model="gpt-4.1", temperature=0.4),
    # "gpt-4-turbo": ChatOpenAI(model="gpt-4-turbo", temperature=0.4),
    # "gpt-4": ChatOpenAI(model="gpt-4-turbo", temperature=0.4),
    # "gpt-3.5-turbo": ChatOpenAI(model="gpt-3.5-turbo", temperature=0.4),
    "gemini-2.5-flash": ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.4,
        api_key=os.environ["GEMINI_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai"
       ),
    "mistral-tiny": ChatOpenAI(
        model="mistral-tiny",
        temperature=0.4,
        api_key=os.environ["MISTRAL_AI_API_KEY"],
        base_url="https://api.mistral.ai/v1",
    ),
}

test_sets = {
    "Test1": {
        "questions": "/content/drive/MyDrive/Colab Notebooks/Stats/QUESTION/Question1.txt",
        "expected": "/content/drive/MyDrive/Colab Notebooks/Stats/ANSWER/ExpectedAnswer1.txt",
    },
    "Test2": {
        "questions": "/content/drive/MyDrive/Colab Notebooks/Stats/QUESTION/Question2.txt",
        "expected": "/content/drive/MyDrive/Colab Notebooks/Stats/ANSWER/ExpectedAnswer2.txt",
    },
    "Test3": {
        "questions": "/content/drive/MyDrive/Colab Notebooks/Stats/QUESTION/Question3.txt",
        "expected": "/content/drive/MyDrive/Colab Notebooks/Stats/ANSWER/ExpectedAnswer3.txt",
    }
}

test_sets_1 = {
    "Test4": {
        "questions": "/content/drive/MyDrive/Colab Notebooks/Stats/QUESTION/Question4.txt",
        "expected": "/content/drive/MyDrive/Colab Notebooks/Stats/ANSWER/ExpectedAnswer4.txt",
    },
    "Test5": {
        "questions": "/content/drive/MyDrive/Colab Notebooks/Stats/QUESTION/Question5.txt",
        "expected": "/content/drive/MyDrive/Colab Notebooks/Stats/ANSWER/ExpectedAnswer5.txt",
    }
}

# Function which extracts generated answers
def extract_generated_answers(file_path):
    answers = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if "RISPOSTA GENERATA:" in line:
                answers.append(line.split("RISPOSTA GENERATA:")[1].strip())
    return answers

In [ ]:
# Loading and cleaning csv files

# CSV progress messages
if os.path.exists(CSV_MESSAGES_OUTPUT):
    print("File mod_progress_output.csv già esistente. Carico quello.\n")
    msg = pd.read_csv(CSV_MESSAGES_OUTPUT)

# CSV progress
if os.path.exists(CSV_PROGRESS_OUTPUT):
    print("File mod_progress_messages_output.csv già esistente. Carico quello.\n")
    pro = pd.read_csv(CSV_PROGRESS_OUTPUT)

# JSON progress output
if os.path.exists(JSON_OUTPUT):
    print("File output.json già esistente. Carico quello.\n")

# Grouping content by progress_id
grouped_msg = msg.groupby("progress_id")["content"].apply(lambda x: "\n".join(x)).reset_index()
grouped = pd.merge(grouped_msg, pro, on="progress_id", how="left")

grouped["full_text"] = (
    grouped["subject"].fillna("") + "\n\n" +
    grouped["description"].fillna("") + "\n\n" +
    grouped["content"].fillna("")
)

In [ ]:
# Splitting documents
with open(JSON_OUTPUT, "r", encoding="utf-8") as f:
    dataset = json.load(f)

docs = []
for progress in dataset:
    header = f"{progress['subject']}\n\n{progress['description']}"
    message_texts = "\n".join(m["content"] for m in progress["messages"])
    full_text = f"{header}\n\n{message_texts}"

    docs.append(Document(
        page_content=full_text,
        metadata={"progress_id": progress["progress_id"]}
    ))

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=60)
chunks = splitter.split_documents(docs)

# Embedding and FAISS fase
embedding_model = OpenAIEmbeddings()
if os.path.exists(FAISS_INDEX_1):
    print("Vector store già esistente. Carico quello.\n")
    vector_store = FAISS.load_local(FAISS_INDEX_1, embedding_model, allow_dangerous_deserialization=True)
else:
    vector_store = FAISS.from_documents(chunks, embedding_model)
    # Saving the vector store
    vector_store.save_local(FAISS_INDEX_1)

In [ ]:
# Prompt and query
prompt = build_prompt_template("boolean")

question = "Nel 2024 sono state attivate 5 piattaforme per il whistleblowing? Rispondi solo con 'True' o 'False'."

# Searching similiraty in docs (reranking optimizing answers)

tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-reranker-base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained("BAAI/bge-reranker-base").to(device)

# Searching focusing on relevance (top-k results)
retrieved_docs = vector_store.max_marginal_relevance_search(question, k=6)
final_docs = rerank(question, retrieved_docs, top_n=3)

docs_content = "\n\n".join(doc.page_content for doc in final_docs)

message = prompt.invoke({"question": question, "context": docs_content, })

answer = llm_models["gpt-4.1"].invoke(message)

printQA(question, answer)
print("Risposta attesa: " + "\n")
print("True")

In [ ]:
# Piece of code where there are some user's questions and expected answers (TRUE or FALSE)

# RAG system now tries to reply at user's questions
generated_answers = []
# Result summary for the % of success
results_summary = []

for test_name, paths in test_sets.items():
    questions = read_lines(paths["questions"])
    expected_answers = read_lines(paths["expected"])
    for model_name, llm in llm_models.items():
        generated_answers = []
        qa_output_path = f"/content/drive/MyDrive/Colab Notebooks/Stats/TEST_RESULTS_MMR/TRUE_FALSE/QA_{test_name}_{model_name}.txt"

        with open(qa_output_path, "w", encoding="utf-8") as f:
            for idx, (question, expected) in enumerate(zip(questions, expected_answers), start=1):
                retrieved_docs = vector_store.max_marginal_relevance_search(question, k=6)
                final_docs = rerank(question, retrieved_docs, top_n=3)

                docs_content = "\n\n".join(doc.page_content for doc in final_docs)
                message = prompt.invoke({"question": question, "context": docs_content})

                if "gemini" in model_name.lower():
                    success = False
                    for attempt in range(5):
                        try:
                            response = llm.invoke(message)
                            success = True
                            break
                        except Exception as e:
                            if "429" in str(e) or "ResourceExhausted" in str(e):
                                wait_time = 6
                                print(f"Quota superata, attendo {wait_time}s...")
                                time.sleep(wait_time)
                            else:
                                raise e

                    if not success:
                        print("Richiesta fallita dopo i retry")
                        continue

                    time.sleep(6)

                else:
                    response = llm.invoke(message)

                generated = response.content.strip()
                generated_answers.append(generated)

                f.write(f"{idx}. DOMANDA: {question}\n")
                f.write(f"   RISPOSTA ATTESA: {expected}\n")
                f.write(f"   RISPOSTA GENERATA: {generated}\n")
                f.write("-" * 80 + "\n")

        # % of success
        success = calculate_success_rate(generated_answers, expected_answers)

        results_summary.append({
            "test": test_name,
            "model": model_name,
            "success": success
        })

correct_answers = read_lines(paths["expected"])

success = calculate_success_rate(generated_answers, correct_answers)

chunk_size = 300
chunk_overlap = 60
retrieval_strategy = "max_marginal_relevance"

# Creating the table of results_summary
with open(CSV_PATH_1, mode="a", newline='', encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)

    if csv_file.tell() == 0:
        writer.writerow(["Timestamp", "Test", "Modello", "Successo (%)", "Chunk Size", "Chunk Overlap", "Retrieval Strategy"])

    for result in results_summary:
        writer.writerow([
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            result["test"],
            result["model"],
            f"{result['success']:.2f}",
            chunk_size,
            chunk_overlap,
            retrieval_strategy
        ])

In [ ]:
# Piece of code where there are some user's questions and expected answers (FREE ANSWERS)
prompt = build_prompt_template()
# RAG system now tries to reply at user's questions
generated_answers = []
# Result summary for the % of success
results_summary = []

for test_name, paths in test_sets_1.items():
    questions = read_lines(paths["questions"])
    expected_answers = read_lines(paths["expected"])
    for model_name, llm in llm_models.items():
        generated_answers = []
        qa_output_path = f"/content/drive/MyDrive/Colab Notebooks/Stats/TEST_RESULTS_MMR/DOMANDE_APERTE_09/QA_{test_name}_{model_name}.txt"

        with open(qa_output_path, "w", encoding="utf-8") as f:
            for idx, (question, expected) in enumerate(zip(questions, expected_answers), start=1):
                retrieved_docs = vector_store.max_marginal_relevance_search(question, k=6)
                final_docs = rerank(question, retrieved_docs, top_n=3)

                docs_content = "\n\n".join(doc.page_content for doc in final_docs)
                message = prompt.invoke({"question": question, "context": docs_content})

                if "gemini" in model_name.lower():
                    success = False
                    for attempt in range(5):
                        try:
                            response = llm.invoke(message)
                            success = True
                            break
                        except Exception as e:
                            if "429" in str(e) or "ResourceExhausted" in str(e):
                                wait_time = 6
                                print(f"Quota superata, attendo {wait_time}s...")
                                time.sleep(wait_time)
                            else:
                                raise e

                    if not success:
                        print("Richiesta fallita dopo i retry")
                        continue

                    time.sleep(6)

                else:
                    response = llm.invoke(message)

                generated = response.content.strip()
                generated_answers.append(generated)

                similarity = semantic_similarity(generated, expected)
                threshold = 0.9
                correct = similarity >= threshold

                f.write(f"{idx}. DOMANDA: {question}\n")
                f.write(f"   RISPOSTA ATTESA: {expected}\n")
                f.write(f"   RISPOSTA GENERATA: {generated}\n")
                f.write(f"   SIMILARITÀ SEMANTICA: {similarity:.2f}\n")
                f.write(f"   CORRETTA? {'✅' if correct else '❌'}\n")
                f.write("-" * 80 + "\n")

In [ ]:
# Results of QA
CSV_SUMMARY="/content/drive/MyDrive/Colab Notebooks/Stats/tab_stats.csv"
RETRIEVAL_STRATEGY = "max-marginal-relevance"

# Folders
true_false_folder = "/content/drive/MyDrive/Colab Notebooks/Stats/TEST_RESULTS_MMR/TRUE_FALSE"
open_questions_base = "/content/drive/MyDrive/Colab Notebooks/Stats/TEST_RESULTS_MMR"

# Soglie per domande aperte
thresholds = [0.5, 0.75, 0.9]

results_tab_stats = []

# TRUE/FALSE
for filename in os.listdir(true_false_folder):
    if not filename.endswith(".txt"):
        continue

    name = filename.replace("QA_", "").replace(".txt", "")
    test_name, model_name = name.split("_", 1)

    file_path = os.path.join(true_false_folder, filename)
    generated_answers = extract_generated_answers(file_path)
    expected_answers = read_lines(test_sets[test_name]["expected"])

    success = calculate_success_rate(generated_answers, expected_answers)

    results_tab_stats.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "test": test_name,
        "model": model_name,
        "type": "TRUE/FALSE",
        "threshold": "",
        "success": success,
        "retrieval": RETRIEVAL_STRATEGY
    })

# DOMANDE_APERTE
for threshold in thresholds:
    folder = f"DOMANDE_APERTE_{str(threshold).replace('.', '')}"
    folder_path = os.path.join(open_questions_base, folder)

    for filename in os.listdir(folder_path):
        if not filename.endswith(".txt"):
            continue

        name = filename.replace("QA_", "").replace(".txt", "")
        test_name, model_name = name.split("_", 1)

        file_path = os.path.join(folder_path, filename)
        generated_answers = extract_generated_answers(file_path)
        expected_answers = read_lines(test_sets_1[test_name]["expected"])

        correct_count = sum(
            1 for g, e in zip(generated_answers, expected_answers)
            if semantic_similarity(g, e) >= threshold
        )
        success = (correct_count / len(expected_answers)) * 100

        results_tab_stats.append({
            "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "test": test_name,
            "model": model_name,
            "type": "DOMANDE_APERTE",
            "threshold": threshold,
            "success": success,
            "retrieval": RETRIEVAL_STRATEGY
        })

with open(CSV_SUMMARY, mode="a", newline='', encoding="utf-8") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "Timestamp", "Test", "Modello", "Tipo Domande",
        "Threshold", "Successo (%)", "Retrieval Strategy"
    ])
    for result in results_tab_stats:
        writer.writerow([
            result["timestamp"],
            result["test"],
            result["model"],
            result["type"],
            result["threshold"],
            f"{result['success']:.2f}",
            result["retrieval"]
        ])

print(f"✅ Tabella tab_stats salvata in: {CSV_SUMMARY}")